# Adım 5: Özellik Mühendisliği
**Beyda sorumluluğu** — `feature/ml-dashboard` branch

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg
from pyspark.sql.window import Window

GOLD_PATH    = './delta_lake/gold'
FEATURE_PATH = './delta_lake/features'

def create_spark():
    return (
        SparkSession.builder
        .appName('ClimateFeatureEngineering')
        .master('local[*]')
        .config('spark.sql.extensions',
                'io.delta.sql.DeltaSparkSessionExtension')
        .config('spark.sql.catalog.spark_catalog',
                'org.apache.spark.sql.delta.catalog.DeltaCatalog')
        .config('spark.jars.packages',
                'io.delta:delta-core_2.12:2.4.0')
        .getOrCreate()
    )

spark = create_spark()
spark.sparkContext.setLogLevel('WARN')
print('[Feature] Gold tablosu okunuyor...')
df = spark.read.format('delta').load(GOLD_PATH)
print(f'[Feature] {df.count():,} kayit yuklendi.')

26/05/12 14:53:05 WARN Utils: Your hostname, Canpolat-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 172.20.10.14 instead (on interface en0)
26/05/12 14:53:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/canpolat/.ivy2/cache
The jars for the packages stored in: /Users/canpolat/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7060c7d5-5f95-40f9-98dc-3307bf7b8f38;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central


:: loading settings :: url = jar:file:/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


:: resolution report :: resolve 86ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-core_2.12;2.4.0 from central in [default]
	io.delta#delta-storage;2.4.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-7060c7d5-5f95-40f9-98dc-3307bf7b8f38
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/3ms)
26/05/12 14:53:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log le

[Feature] Gold tablosu okunuyor...


26/05/12 14:53:09 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[Feature] 15,882,533 kayit yuklendi.


In [2]:
# sicaklik farki: gunluk max ile min arasindaki fark
df = df.withColumn('temp_range', col('max_temp_c') - col('min_temp_c'))

# mevsimi sayiya donusturuyoruz
df = df.withColumn(
    'season_num',
    when(col('season') == 'Winter', 0)
    .when(col('season') == 'Spring', 1)
    .when(col('season') == 'Summer', 2)
    .when(col('season') == 'Autumn', 3)
    .otherwise(0)
)

# ay sutunu zaten var, kontrol et
print('Features 1-3 eklendi: temp_range, season_num, month')
df.select('station_id', 'date', 'avg_temp_c', 'temp_range', 'season_num', 'month').show(5)

Features 1-3 eklendi: temp_range, season_num, month
+----------+-------------------+----------+-----------------+----------+-----+
|station_id|               date|avg_temp_c|       temp_range|season_num|month|
+----------+-------------------+----------+-----------------+----------+-----+
|     01008|1975-10-23 00:00:00|      -5.1|              4.9|         3|   10|
|     01008|1975-12-17 00:00:00|     -20.9|5.400000000000002|         0|   12|
|     01008|1976-02-03 00:00:00|     -16.7|5.600000000000001|         0|    2|
|     01008|1976-02-14 00:00:00|      -1.8|             11.7|         0|    2|
|     01008|1976-02-21 00:00:00|      -7.7|             27.0|         0|    2|
+----------+-------------------+----------+-----------------+----------+-----+
only showing top 5 rows



## Feature 4-6 ve Kaydetme

In [3]:
# son 7 gunun ortalama sicakligi
window_spec = (
    Window.partitionBy('station_id')
    .orderBy('date')
    .rowsBetween(-6, 0)
)
df = df.withColumn('rolling_avg_7', avg('avg_temp_c').over(window_spec))

# cok sicak veya cok soguk gunleri isaretliyoruz
df = df.withColumn(
    'is_extreme_temp',
    when((col('avg_temp_c') > 35) | (col('avg_temp_c') < -20), 1).otherwise(0)
)

# yagisi kategorilere ayiriyoruz
df = df.withColumn(
    'prcp_category',
    when(col('precipitation_mm').isNull() | (col('precipitation_mm') == 0), 0)
    .when(col('precipitation_mm') <= 5, 1)
    .when(col('precipitation_mm') <= 20, 2)
    .otherwise(3)
)

print('Features 4-6 eklendi: rolling_avg_7, is_extreme_temp, prcp_category')

Features 4-6 eklendi: rolling_avg_7, is_extreme_temp, prcp_category


In [4]:
# ozellikleri delta lake e kaydediyoruz
feature_cols = [
    'station_id', 'city_name', 'date', 'year', 'month',
    'avg_temp_c',
    'temp_range', 'season_num', 'rolling_avg_7',
    'is_extreme_temp', 'prcp_category',
    'avg_wind_speed_kmh', 'avg_sea_level_pres_hpa', 'sunshine_total_min',
]
df_feat = df.select(feature_cols).dropna(subset=['avg_temp_c', 'rolling_avg_7'])
count = df_feat.count()
print(f'[Feature] Kayit sayisi: {count:,}')
df_feat.write.format('delta').mode('overwrite').save(FEATURE_PATH)
print(f'[Feature] Ozellik tablosu yazildi -> {FEATURE_PATH}')
df_feat.show(5, truncate=False)
spark.stop()
print('Ozellik muhendisligi tamamlandi.')

[Feature] Kayit sayisi: 15,882,533


26/05/12 14:53:28 WARN MemoryManager: Total allocation exceeds 95,00% (1.020.054.720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


[Feature] Ozellik tablosu yazildi -> ./delta_lake/features


+----------+---------+-------------------+----+-----+----------+------------------+----------+------------------+---------------+-------------+------------------+----------------------+------------------+
|station_id|city_name|date               |year|month|avg_temp_c|temp_range        |season_num|rolling_avg_7     |is_extreme_temp|prcp_category|avg_wind_speed_kmh|avg_sea_level_pres_hpa|sunshine_total_min|
+----------+---------+-------------------+----+-----+----------+------------------+----------+------------------+---------------+-------------+------------------+----------------------+------------------+
|02590     |Visby    |1977-07-01 00:00:00|1977|7    |16.2      |7.299999999999999 |2         |16.2              |0              |2            |null              |null                  |null              |
|02590     |Visby    |1977-07-02 00:00:00|1977|7    |12.3      |6.899999999999999 |2         |14.25             |0              |2            |null              |null              